# ISCE3 and CosiCorr or Micmac Run Notebook

## Get the slc from the RSLC with geocoding by running isce3 processing

In [1]:
import subprocess
import sys

# RAN ONE SET AT A TIME, UPDATED yaml individually

# Path to yaml ASCENDING
# To do, pass DEM, and RSLC names to a function that updates the yaml automatically
# To do, instead of running offsets or insar, just run GSLC processing workflow
reference_rslc_file = '/Users/chanagan/Downloads/Panama2026/NISAR_ASF/RSLC/NISAR_L1_PR_RSLC_032_076_A_005_4005_DHDH_A_20261005T105604_20261005T105616_P05023_N_P_J_001.h5'
secondary_rslc_file = '/Users/chanagan/Downloads/Panama2026/NISAR_ASF/RSLC/NISAR_L1_UR_RSLC_033_076_A_005_4005_DHDH_A_20261017T105604_20261017T105616_P05023_F_P_J_001.h5'
yaml_file = "/Users/chanagan/Documents/GitHub/ImageryResources/ISCE3_Workflows/NISAR_dense_offsets.yaml"
#scratch_path: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratchA/

# Path to yaml DESCENDING
reference_rslc_file = '/Users/chanagan/Downloads/Panama2026/NISAR_ASF/RSLC/NISAR_L1_PR_RSLC_031_141_D_085_4005_DHDH_A_20260927T235335_20260927T235344_P05023_N_P_J_001.h5'
secondary_rslc_file = '/Users/chanagan/Downloads/Panama2026/NISAR_ASF/RSLC/NISAR_L1_PR_RSLC_032_141_D_085_4005_DHDH_A_20261009T235335_20261009T235345_P05023_N_P_J_001.h5'
yaml_file = "/Users/chanagan/Documents/GitHub/ImageryResources/ISCE3_Workflows/NISAR_dense_offsets.yaml"
#scratch_path: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratchDw/

# Path to yaml DESCENDING
reference_rslc_file = '/Users/chanagan/Downloads/Panama2026/NISAR_ASF/RSLC/NISAR_L1_PR_RSLC_032_040_D_085_4005_DHDH_A_20261002T234516_20261002T234533_P05023_N_P_J_001.h5'
secondary_rslc_file = '/Users/chanagan/Downloads/Panama2026/NISAR_ASF/RSLC/NISAR_L1_PR_RSLC_033_040_D_085_4005_DHDH_A_20261014T234516_20261014T234533_P05023_N_P_J_001.h5'
yaml_file = "/Users/chanagan/Documents/GitHub/ImageryResources/ISCE3_Workflows/NISAR_dense_offsets.yaml"
#scratch_path: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratchDe/
# Path to yaml ASCENDING


In [ ]:
cmd = [
    sys.executable,
    "-m",
    "nisar.workflows.insar",
    yaml_file
]

subprocess.run(cmd, check=True)

## Convert slc to tif

In [4]:
import rasterio
import re 
import os
import numpy as np
 
# Path with isce3 files
isce3Path = '/Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratchDe'

os.chdir(isce3Path)

files = [
     (
         "./dense_offsets/freqA/HH/reference.slc",
         "./dense_offsets/freqA/HH/reference_amplitude.tif",
     ),
     (
         "./coarse_resample_slc/freqA/HH/coregistered_secondary.slc",
         "./coarse_resample_slc/freqA/HH/coregistered_secondary_amplitude.tif",
     ),
 ]

for slc_file, out_file in files:
    hdr_file = os.path.splitext(slc_file)[0] + ".hdr"

    with open(hdr_file) as f:
        header = f.read()

    width = int(re.search(r"samples\s*=\s*(\d+)", header, re.I).group(1))
    height = int(re.search(r"lines\s*=\s*(\d+)", header, re.I).group(1))
    dtype_code = int(re.search(r"data type\s*=\s*(\d+)", header, re.I).group(1))

    if dtype_code == 6:
        dtype = np.dtype("<c8")
    elif dtype_code == 4:
        dtype = np.dtype("<f4")
    else:
        raise ValueError(f"Unexpected ENVI data type {dtype_code}: {slc_file}")

    slc = np.memmap(
        slc_file, dtype=dtype, mode="r",
        shape=(height, width),
    )

    profile = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": 1,
        "dtype": "float32",
        "compress": "LZW",
        "zlevel":9,
        "BIGTIFF": "IF_SAFER",
    }

    with rasterio.open(out_file, "w", **profile) as dst:
        for row in range(0, height, 256):
            end = min(row + 256, height)
            amplitude = np.abs(slc[row:end]).astype(np.float32)
            dst.write(amplitude, 1, window=((row, end), (0, width)))

    del slc
    print(f"Saved {out_file}: {width} x {height}")
   

/Users/chanagan/.local/envs/isce3/lib/python3.14/site-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(


Saved ./dense_offsets/freqA/HH/reference_amplitude.tif: 52475 x 27360
Saved ./coarse_resample_slc/freqA/HH/coregistered_secondary_amplitude.tif: 52475 x 27360


## ASP image align

In [ ]:
conda activate asp

cd /Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratchDe

image_align \
  ./dense_offsets/freqA/HH/reference_amplitude.tif \
    ./coarse_resample_slc/freqA/HH/coregistered_secondary_amplitude.tif \
  --alignment-transform translation \
  --output-prefix ./align/run \
  -o ./coarse_resample_slc/freqA/HH/secondary_amplitude_align.tif




CondaError: Run 'conda init' before 'conda activate'

bash: line 7: image_align: command not found


CalledProcessError: Command 'b'\nconda activate asp\n\ncd /Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratchDe\n\nimage_align \\\n  ./dense_offsets/freqA/HH/reference_amplitude.tif \\\n    ./coarse_resample_slc/freqA/HH/coregistered_secondary_amplitude.tif \\\n  --alignment-transform translation \\\n  --output-prefix align/run \\\n  -o ./coarse_resample_slc/freqA/HH/secondary_amplitude_align.tif\n\n'' returned non-zero exit status 127.

### Clip tif

In [5]:
import rasterio
from rasterio.windows import Window
import os
 
# Path with isce3 files
isce3Path = '/Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratchDe'

os.chdir(isce3Path)

reference_file = "./dense_offsets/freqA/HH/reference_amplitude.tif"
secondary_file = "./coarse_resample_slc/freqA/HH/coregistered_secondary_amplitude.tif"

reference_clip = "./dense_offsets/freqA/HH/reference_amplitude_clip.tif"
secondary_clip = "./coarse_resample_slc/freqA/HH/secondary_amplitude_clip.tif"

# Crop bounds in original pixel coordinates
# ASCENDING
col_start, col_end = 34963, 49895
row_start, row_end = 6156, 17862
# DESCENDING W
col_start, col_end = 40385, 52568
row_start, row_end = 3156, 15195
# DESCENDING E
col_start, col_end = 0, 20681
row_start, row_end = 0, 26476


window = Window(
    col_start,
    row_start,
    col_end - col_start,
    row_end - row_start
)

for input_file, output_file in [
    (reference_file, reference_clip),
    (secondary_file, secondary_clip)
]:
    with rasterio.open(input_file) as src:
        profile = src.profile.copy()
        profile.update(
            width=int(window.width),
            height=int(window.height)
        )

        with rasterio.open(output_file, "w", **profile) as dst:
            dst.write(src.read(window=window))


/Users/chanagan/.local/envs/isce3/lib/python3.14/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/Users/chanagan/.local/envs/isce3/lib/python3.14/site-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


### Use transform to align

In [1]:
import numpy as np
import cv2
import rasterio
import os

os.chdir('/Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratchDe')

# Inputs
src_file = "./coarse_resample_slc/freqA/HH/secondary_amplitude_clip.tif"
ref_file = "./dense_offsets/freqA/HH/reference_amplitude_clip.tif"
transform_file = "./align/run-transform.txt"
out_file = "./coarse_resample_slc/freqA/HH/secondary_amplitude_clip_align_nearest.tif"

# Read transform (3x3 matrix)
M = np.loadtxt(transform_file)

# Read source and reference metadata
with rasterio.open(src_file) as src:
    image = src.read(1)
    nodata = src.nodata if src.nodata is not None else -9999

with rasterio.open(ref_file) as ref:
    profile = ref.profile.copy()
    width = ref.width
    height = ref.height

# Apply source-to-reference transform using nearest neighbor
aligned = cv2.warpPerspective(
    image,
    M,
    (width, height),
    flags=cv2.INTER_NEAREST,
    borderMode=cv2.BORDER_CONSTANT,
    borderValue=nodata
)

# Write output on full reference grid
profile.update(
    driver="GTiff",
    width=width,
    height=height,
    count=1,
    dtype="float32",
    nodata=nodata,
    compress="LZW",
    BIGTIFF="YES"
)

with rasterio.open(out_file, "w", **profile) as dst:
    dst.write(aligned.astype(np.float32), 1)

/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


## Correlate with desired correlator

e.g. CosiCorr in docker:

```
# Has to be run from cosicorr github repo folder
docker run --rm -it \
  --platform linux/amd64 \
  --mount "type=bind,source=$PWD,target=/work" \
  -v /Users/chanagan/Downloads:/home/user/Downloads \
  --workdir /work \
  geocosicorr3d:latest \
  bash

python3 scripts/cosicorr.py correlate --step 1 1 ../home/user/Downloads/reference_amplitude.tif \
    ../home/user/Downloads/coregistered_secondary_amplitude.tif 
```

e.g. MicMac in docker

```
# Can be run from folder with files
docker run --rm -it \
  --platform linux/amd64 \
  --mount "type=bind,source=$PWD,target=/work" \
  --workdir /work \
  rupnike/micmac:1.0 \
  bash

mm3d Mm2dPosSism reference_amplitude.tif coregistered_secondary_amplitude.tif \
    Inc=10 Dequant=false DirMEC='MEC_SLCInc10/'
```


In [ ]:
## CURRENT WIP
# --------------------------------------------------
# Select correlator and configuration
# --------------------------------------------------
import os
import h5py
import numpy as np
import rasterio

from osgeo import gdal
from pyproj import Geod
from scipy.ndimage import zoom
from pyproj import Transformer

import sys
sys.path.append('/Users/chanagan/Documents/GitHub/imagery-harmonization/')
import harmonization as hz

correlator = 'micmac'  # micmac or cosicorr

cropped = True

if cropped == True:
    # Original crop coordinates
    col_start, col_end = 34963, 49895
    row_start, row_end = 6156, 17862

work_dir = "/Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratch"
os.chdir(work_dir)

dispalcement_folder = 'MEC_SLC_clipAligned_inc4resopt1'

if correlator == 'cosicorr':
    displacement_tif = (
    f"{work_dir}/reference_amplitude_VS_coregistered_secondary_amplitude_frequency_wz_64_step_8.tif"
    )

geom_dir = f"{work_dir}/rdr2geo/freqA"

reference_tif = (
    f"{work_dir}/dense_offsets/freqA/HH/reference_amplitude.tif"
)

rslc_file = (
    f"../RSLC/"
    "NISAR_L1_PR_RSLC_032_076_A_005_4005_DHDH_A_"
    "20261005T105604_20261005T105616_P05023_N_P_J_001.h5"
)

nodata = -9999.0

output_crs = "EPSG:32617"
resolution = 5
output_bounds = [497979,793408,576316,854849] # [xmin, ymin, xmax, ymax], or None

# keep land only
landmask_file = '/Users/chanagan/Library/CloudStorage/OneDrive-DOI/Data/MaskFiles/gshhg-shp-2.3.7/GSHHS_shp/f/GSHHS_f_L1_epsg32617.geojson'        

# Geolocation subsampling
step = 4

# Azimuth-spacing sampling
azimuth_sample_step = 500


# --------------------------------------------------
# Correlator-specific inputs
# --------------------------------------------------
if correlator == 'cosicorr':

    corrfactor = 1.0

    with rasterio.open(reference_tif) as ref:
        width = ref.width
        height = ref.height

    offsets = {}

    with rasterio.open(displacement_tif) as src:

        source_nodata = src.nodata

        for band, name in enumerate(["X", "Y", "SNR"], start=1):

            data = src.read(band).astype(np.float32)

            # Upsample to reference dimensions
            output = zoom(
                data,
                (height / src.height, width / src.width),
                order=1
            ).astype(np.float32)

            output = output[:height, :width]

            if output.shape != (height, width):
                raise ValueError(
                    f"{name} dimensions {output.shape} do not match "
                    f"reference dimensions {(height, width)}"
                )

            offsets[name] = output

    offset_nodata = {
        "X": source_nodata,
        "Y": source_nodata,
        "SNR": source_nodata
    }

elif correlator == 'micmac': 

    corrfactor = 0.05

    # Original and clipped reference dimensions
    with rasterio.open(reference_tif) as ref, \
         rasterio.open(
             f"{work_dir}/dense_offsets/freqA/HH/reference_amplitude_clip.tif"
         ) as clipped:

        width = ref.width
        height = ref.height

        crop_width = clipped.width
        crop_height = clipped.height

    # MicMac displacement files
    range_file = (
        f"{work_dir}/{dispalcement_folder}/"
        "Px1_Num5_DeZoom1_LeChantier.tif"
    )

    azimuth_file = (
        f"{work_dir}/{dispalcement_folder}/"
        "Px2_Num5_DeZoom1_LeChantier.tif"
    )    
    
    corr_file = (
        f"{work_dir}/{dispalcement_folder}/"
        "Correl_LeChantier_Num_5.tif"
    )

    # Restored full-size displacement files
    restored_range = (
        f"{work_dir}/{dispalcement_folder}/"
        "RangeDisplacementMicmac.tif"
    )

    restored_azimuth = (
        f"{work_dir}/{dispalcement_folder}/"
        "AzimuthDisplacementMicmac.tif"
    )

    restored_correl = (
        f"{work_dir}/{dispalcement_folder}/"
        "CorrelMicmac.tif"
    )

    offsets = {}

    for input_file, output_file, name in [
        (range_file, restored_range, "X"),
        (azimuth_file, restored_azimuth, "Y"),
        (corr_file, restored_correl,"Corr")
    ]:

        with rasterio.open(input_file) as src:
            data = np.flip(src.read(1).astype(np.float32), axis=0)

            assert src.width == crop_width
            assert src.height == crop_height

            source_nodata = src.nodata

            # Preserve the vertical orientation correction
            data = np.flip(data, axis=0)

            # Initialize full-size radar-grid array
            restored = np.full(
                (height, width),
                nodata,
                dtype=np.float32
            )

            # Insert clipped displacement at original pixel coordinates
            restored[
                row_start:row_end,
                col_start:col_end
            ] = data

            if source_nodata is not None:
                restored[
                    row_start:row_end,
                    col_start:col_end
                ][data == source_nodata] = nodata

            # Write restored TIFF
            with rasterio.open(
                output_file,
                "w",
                driver="GTiff",
                width=width,
                height=height,
                count=1,
                dtype="float32",
                nodata=nodata,
                compress="LZW"
            ) as dst:

                dst.write(restored, 1)

            offsets[name] = restored

            if name == "X":
                range_nodata = nodata
            else:
                azimuth_nodata = nodata

    #offsets["Y"] *= -1 # micmac additional multiplier for y dir, flipped it, so not needed

    offset_nodata = {
        "X": range_nodata,
        "Y": azimuth_nodata
    }

else:
    raise ValueError(f"Unknown correlator: {correlator}")


# --------------------------------------------------
# Output directory
# --------------------------------------------------
output_dir = f"{work_dir}/{dispalcement_folder}/geocoded"
os.makedirs(output_dir, exist_ok=True)

print(f"Radar-grid dimensions: {width} x {height}")


# --------------------------------------------------
# Verify geolocation dimensions
# --------------------------------------------------
lon_file = f"{geom_dir}/x.rdr"
lat_file = f"{geom_dir}/y.rdr"

assert os.path.exists(lon_file)
assert os.path.exists(lat_file)

lon_ds = gdal.Open(lon_file)
lat_ds = gdal.Open(lat_file)

if lon_ds is None or lat_ds is None:
    raise RuntimeError("Could not open ISCE3 geolocation files")

assert lon_ds.RasterXSize == width
assert lon_ds.RasterYSize == height
assert lat_ds.RasterXSize == width
assert lat_ds.RasterYSize == height


# --------------------------------------------------
# Derive slant-range spacing from RSLC metadata
# --------------------------------------------------
with h5py.File(rslc_file, "r") as f:

    range_spacing = float(
        f[
            "/science/LSAR/RSLC/swaths/frequencyA/"
            "slantRangeSpacing"
        ][()]
    )

range_factor = corrfactor * range_spacing

print(f"Slant-range spacing: {range_spacing:.6f} m/pixel")


# --------------------------------------------------
# Derive ground-projected azimuth spacing
# --------------------------------------------------
# Read full geolocation arrays, as in the original
# working workflow.
lon_full = lon_ds.ReadAsArray()
lat_full = lat_ds.ReadAsArray()

assert lon_full.shape == (height, width)
assert lat_full.shape == (height, width)

# Subsample for azimuth-spacing calculation
lon = lon_full[::azimuth_sample_step, ::azimuth_sample_step]
lat = lat_full[::azimuth_sample_step, ::azimuth_sample_step]

lon1 = lon[:-1, :]
lat1 = lat[:-1, :]

lon2 = lon[1:, :]
lat2 = lat[1:, :]

valid = (
    np.isfinite(lon1) & np.isfinite(lat1) &
    np.isfinite(lon2) & np.isfinite(lat2) &
    (np.abs(lon1) <= 180) & (np.abs(lon2) <= 180) &
    (np.abs(lat1) <= 90) & (np.abs(lat2) <= 90)
)

geod = Geod(ellps="WGS84")

_, _, distances = geod.inv(
    lon1[valid],
    lat1[valid],
    lon2[valid],
    lat2[valid]
)

distances = distances[
    np.isfinite(distances) & (distances > 0)
]

if distances.size == 0:
    raise RuntimeError("Could not calculate azimuth pixel spacing")

azimuth_spacing = np.median(distances) / azimuth_sample_step

print("Azimuth spacing std:", np.std(distances / azimuth_sample_step))

azimuth_factor = corrfactor * azimuth_spacing

print(f"Ground-projected azimuth spacing: {azimuth_spacing:.6f} m/pixel")
print(f"Range conversion factor: {range_factor:.6f} m/unit")
print(f"Azimuth conversion factor: {azimuth_factor:.6f} m/unit")

# --------------------------------------------------
# Create subsampled geolocation GeoTIFF
# --------------------------------------------------
lon_geo = lon_full[::step, ::step].astype(np.float32)
lat_geo = lat_full[::step, ::step].astype(np.float32)

geolocation_file = f"{output_dir}/geolocation.tif"

transformer = Transformer.from_crs(
    "EPSG:4326",
    output_crs,
    always_xy=True
)

x, y = transformer.transform(lon_geo, lat_geo)

valid = (
    np.isfinite(x) &
    np.isfinite(y) &
    (lon_geo >= -180) & (lon_geo <= 180) &
    (lat_geo >= -90) & (lat_geo <= 90)
)

geo_bounds = (
    float(np.min(x[valid])),
    float(np.min(y[valid])),
    float(np.max(x[valid])),
    float(np.max(y[valid]))
)

print("Geolocation bounds (UTM 17N):", geo_bounds)
print("Geolocation dimensions:", lon_geo.shape)

geo_profile = {
    "driver": "GTiff",
    "height": lon_geo.shape[0],
    "width": lon_geo.shape[1],
    "count": 2,
    "dtype": "float32",
    "compress": "LZW",
    "tiled": True,
    "BIGTIFF": "IF_SAFER"
}

with rasterio.open(geolocation_file, "w", **geo_profile) as dst:
    dst.write(lon_geo, 1)
    dst.write(lat_geo, 2)

print(f"Created geolocation grid: {lon_geo.shape}")

# Release large geolocation arrays
del lon_full, lat_full
del lon, lat, lon_geo, lat_geo

lon_ds = None
lat_ds = None


# --------------------------------------------------
# Create three-band in-memory displacement raster
# --------------------------------------------------
if correlator == 'micmac':
    band_names = ["X", "Y", "Corr"]
else:
    band_names = ["X", "Y", "SNR"]

factors = {
    "X": range_factor,
    "Y": azimuth_factor,
    band_names[-1]: 1.0
}

mem = gdal.GetDriverByName("MEM").Create(
    "",
    width,
    height,
    len(band_names),
    gdal.GDT_Float32
)

for i, name in enumerate(band_names, start=1):

    if name not in offsets:
        continue

    data = offsets[name]
    factor = factors[name]

    if data.shape != (height, width):
        raise ValueError(
            f"{name} dimensions {data.shape} do not match "
            f"reference dimensions {(height, width)}"
        )

    valid = np.isfinite(data)

    src_nodata = offset_nodata.get(name)

    if src_nodata is not None:
        valid &= data != src_nodata

    output = np.full(
        (height, width),
        nodata,
        dtype=np.float32
    )

    output[valid] = data[valid] * factor

    band = mem.GetRasterBand(i)
    band.WriteArray(output)
    band.SetNoDataValue(nodata)

    print(f"Prepared band {i}: {name}")


# --------------------------------------------------
# Attach original subsampled geolocation convention
# --------------------------------------------------
mem.SetMetadata({
    "X_DATASET": geolocation_file,
    "X_BAND": "1",
    "Y_DATASET": geolocation_file,
    "Y_BAND": "2",
    "SRS": "EPSG:4326",
    "PIXEL_OFFSET": "0",
    "LINE_OFFSET": "0",
    "PIXEL_STEP": str(step),
    "LINE_STEP": str(step),
    "GEOREFERENCING_CONVENTION": "PIXEL_CENTER"
}, "GEOLOCATION")


# --------------------------------------------------
# Geocode all bands in one GDAL operation
# --------------------------------------------------
combined_file = f"{output_dir}/displacements_utm17.tif"

print("Geocoding all bands...")
    
warp_options = gdal.WarpOptions(
    format="GTiff",
    geoloc=True,
    dstSRS=output_crs,
    #outputBounds=geo_bounds,
    #cutlineDSName=landmask_file,
    #cropToCutline=False,
    xRes=resolution,
    yRes=resolution,
    targetAlignedPixels=True,
    resampleAlg="nearest",
    srcNodata=nodata,
    dstNodata=nodata,
    multithread=True,
    warpMemoryLimit=1024,
    outputBounds=output_bounds,
    errorThreshold=0.0,
    warpOptions=["NUM_THREADS=ALL_CPUS"],
    creationOptions=[
        "COMPRESS=LZW",
        "TILED=YES",
        "BIGTIFF=IF_SAFER"
    ]
)

gdal.Warp(
    combined_file,
    mem,
    options=warp_options
)

mem = None

print(f"Created: {combined_file}")
# --------------------------------------------------
# Apply ocean mask and write new GeoTIFF
# --------------------------------------------------
masked_file = f"{output_dir}/displacements_utm17_landmasked.tif"

with rasterio.open(combined_file) as src:

    profile = src.profile.copy()

    with rasterio.open(masked_file, "w", **profile) as dst:

        for i in range(1, src.count + 1):

            data = src.read(i)

            data_masked, _ = hz.mask_ocean(
                data,
                src.transform,
                src.crs,
                landmask_file,
                nodata=nodata,
                coastline_buffer=5
            )

            dst.write(
                data_masked,
                i
            )

print(f"Created: {masked_file}")
os.remove(combined_file)
print("Finished masking ocean.")

/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = writer(


Radar-grid dimensions: 52484 x 19760
Slant-range spacing: 3.122838 m/pixel


/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Azimuth spacing std: 0.011755063701617323
Ground-projected azimuth spacing: 4.446995 m/pixel
Range conversion factor: 0.156142 m/unit
Azimuth conversion factor: 0.222350 m/unit
Geolocation bounds (UTM 17N): (487558.0486364624, 759499.0958532026, 752715.5225192403, 902432.7555143941)
Geolocation dimensions: (4940, 13121)
Created geolocation grid: (4940, 13121)
Prepared band 1: X
Prepared band 2: Y
Prepared band 3: Corr
Geocoding all bands...


ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [
ERROR 1: missing [


Created: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratch/MEC_SLC_clipAligned_inc4resopt1/geocoded/displacements_utm17.tif


/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 2 will not be rasterized.
  warnings.warn(
/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 199 will not be rasterized.
  warnings.warn(
/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 332 will not be rasterized.
  warnings.warn(
/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 861 will not be rasterized.
  warnings.warn(
/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 1021 will not be rasterized

Created: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratch/MEC_SLC_clipAligned_inc4resopt1/geocoded/displacements_utm17_landmasked.tif
Finished masking ocean.


In [10]:
data

array([[-9999., -9999., -9999., ..., -9999., -9999., -9999.],
       [-9999., -9999., -9999., ..., -9999., -9999., -9999.],
       [-9999., -9999., -9999., ..., -9999., -9999., -9999.],
       ...,
       [-9999., -9999., -9999., ..., -9999., -9999., -9999.],
       [-9999., -9999., -9999., ..., -9999., -9999., -9999.],
       [-9999., -9999., -9999., ..., -9999., -9999., -9999.]],
      shape=(28585, 53032), dtype=float32)

In [16]:
# --------------------------------------------------
# Apply ocean mask and write new GeoTIFF
# --------------------------------------------------
masked_file = f"{output_dir}/displacements_utm17_landmasked.tif"

with rasterio.open(combined_file) as src:

    profile = src.profile.copy()

    with rasterio.open(masked_file, "w", **profile) as dst:

        for i in range(1, src.count + 1):

            data = src.read(i)

            data_masked, _ = hz.mask_ocean(
                data,
                src.transform,
                src.crs,
                landmask_file,
                nodata=nodata,
                coastline_buffer=5
            )

            dst.write(
                data_masked,
                i
            )

print(f"Created: {masked_file}")

/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 2 will not be rasterized.
  warnings.warn(
/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 199 will not be rasterized.
  warnings.warn(
/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 332 will not be rasterized.
  warnings.warn(
/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 861 will not be rasterized.
  warnings.warn(
/Users/chanagan/miniforge3/envs/ImageryResources/lib/python3.14/site-packages/rasterio/features.py:364: ShapeSkipWarning: Invalid or empty shape None at index 1021 will not be rasterized

Created: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/isce3_scratch/MEC_SLC_clipInc10/geocoded/displacements_utm17_landmasked.tif


In [3]:
data_masked

NameError: name 'data_masked' is not defined

## Next block?

In [ ]:
# --------------------------------------------------
# Select correlator and configuration
# --------------------------------------------------
import os
import h5py
import numpy as np
import rasterio

from osgeo import gdal
from pyproj import Geod
from scipy.ndimage import zoom
from pyproj import Transformer



correlator = 'micmac'  # micmac or cosicorr

cropped = True

if cropped == True:
    # Original crop coordinates
    col_start, col_end = 34963, 49895
    row_start, row_end = 6156, 17862

work_dir = "/Users/chanagan/Downloads/Panama2026/NISAR_ASF"
os.chdir(work_dir)

dispalcement_folder = 'micmac'
displacement_tif = (
    f"{work_dir}/{dispalcement_folder}/reference_amplitude_VS_coregistered_secondary_amplitude_frequency_wz_64_step_8.tif"
)

geom_dir = f"{work_dir}/rdr2geo/freqA"

reference_tif = (
    f"{work_dir}/dense_offsets/freqA/HH/reference_amplitude.tif"
)

rslc_file = (
    f"{work_dir}/RSLC/"
    "NISAR_L1_PR_RSLC_032_076_A_005_4005_DHDH_A_"
    "20261005T105604_20261005T105616_P05023_N_P_J_001.h5"
)

nodata = -9999.0

output_crs = "EPSG:32617"
resolution = 10

# Geolocation subsampling
step = 8

# Azimuth-spacing sampling
azimuth_sample_step = 500


# --------------------------------------------------
# Correlator-specific inputs
# --------------------------------------------------
if correlator == 'cosicorr':

    corrfactor = 1.0

    with rasterio.open(reference_tif) as ref:
        width = ref.width
        height = ref.height

    offsets = {}

    with rasterio.open(displacement_tif) as src:

        source_nodata = src.nodata

        for band, name in enumerate(["X", "Y", "SNR"], start=1):

            data = src.read(band).astype(np.float32)

            # Upsample to reference dimensions
            output = zoom(
                data,
                (height / src.height, width / src.width),
                order=1
            ).astype(np.float32)

            output = output[:height, :width]

            if output.shape != (height, width):
                raise ValueError(
                    f"{name} dimensions {output.shape} do not match "
                    f"reference dimensions {(height, width)}"
                )

            offsets[name] = output

    offset_nodata = {
        "X": source_nodata,
        "Y": source_nodata,
        "SNR": source_nodata
    }

elif correlator == 'micmac':

    corrfactor = 0.05

    # Update these paths to your MicMac displacement rasters
    range_file = f"{work_dir}/{dispalcement_folder}/..._X.tif"
    azimuth_file = f"{work_dir}/{dispalcement_folder}/..._Y.tif"

    offsets = {}

    with rasterio.open(range_file) as src:
        offsets["X"] = src.read(1).astype(np.float32)
        range_nodata = src.nodata
        width = src.width
        height = src.height

    with rasterio.open(azimuth_file) as src:
        offsets["Y"] = -1*src.read(1).astype(np.float32)
        azimuth_nodata = src.nodata

    offset_nodata = {
        "X": range_nodata,
        "Y": azimuth_nodata
    }

else:
    raise ValueError(f"Unknown correlator: {correlator}")


# --------------------------------------------------
# Output directory
# --------------------------------------------------
output_dir = f"{work_dir}/{dispalcement_folder}/geocoded"
os.makedirs(output_dir, exist_ok=True)

print(f"Radar-grid dimensions: {width} x {height}")


# --------------------------------------------------
# Verify geolocation dimensions
# --------------------------------------------------
lon_file = f"{geom_dir}/x.rdr"
lat_file = f"{geom_dir}/y.rdr"

assert os.path.exists(lon_file)
assert os.path.exists(lat_file)

lon_ds = gdal.Open(lon_file)
lat_ds = gdal.Open(lat_file)

if lon_ds is None or lat_ds is None:
    raise RuntimeError("Could not open ISCE3 geolocation files")

assert lon_ds.RasterXSize == width
assert lon_ds.RasterYSize == height
assert lat_ds.RasterXSize == width
assert lat_ds.RasterYSize == height


# --------------------------------------------------
# Derive slant-range spacing from RSLC metadata
# --------------------------------------------------
with h5py.File(rslc_file, "r") as f:

    range_spacing = float(
        f[
            "/science/LSAR/RSLC/swaths/frequencyA/"
            "slantRangeSpacing"
        ][()]
    )

range_factor = corrfactor * range_spacing

print(f"Slant-range spacing: {range_spacing:.6f} m/pixel")


# --------------------------------------------------
# Derive ground-projected azimuth spacing
# --------------------------------------------------
# Read full geolocation arrays, as in the original
# working workflow.
lon_full = lon_ds.ReadAsArray()
lat_full = lat_ds.ReadAsArray()

assert lon_full.shape == (height, width)
assert lat_full.shape == (height, width)

# Subsample for azimuth-spacing calculation
lon = lon_full[::azimuth_sample_step, ::azimuth_sample_step]
lat = lat_full[::azimuth_sample_step, ::azimuth_sample_step]

lon1 = lon[:-1, :]
lat1 = lat[:-1, :]

lon2 = lon[1:, :]
lat2 = lat[1:, :]

valid = (
    np.isfinite(lon1) & np.isfinite(lat1) &
    np.isfinite(lon2) & np.isfinite(lat2) &
    (np.abs(lon1) <= 180) & (np.abs(lon2) <= 180) &
    (np.abs(lat1) <= 90) & (np.abs(lat2) <= 90)
)

geod = Geod(ellps="WGS84")

_, _, distances = geod.inv(
    lon1[valid],
    lat1[valid],
    lon2[valid],
    lat2[valid]
)

distances = distances[
    np.isfinite(distances) & (distances > 0)
]

if distances.size == 0:
    raise RuntimeError("Could not calculate azimuth pixel spacing")

azimuth_spacing = np.median(distances) / azimuth_sample_step

print("Azimuth spacing std:", np.std(distances / azimuth_sample_step))

azimuth_factor = corrfactor * azimuth_spacing

print(f"Ground-projected azimuth spacing: {azimuth_spacing:.6f} m/pixel")
print(f"Range conversion factor: {range_factor:.6f} m/unit")
print(f"Azimuth conversion factor: {azimuth_factor:.6f} m/unit")


# --------------------------------------------------
# Create subsampled geolocation GeoTIFF
# --------------------------------------------------
lon_geo = lon_full[::step, ::step].astype(np.float32)
lat_geo = lat_full[::step, ::step].astype(np.float32)

geolocation_file = f"{output_dir}/geolocation.tif"

transformer = Transformer.from_crs(
    "EPSG:4326",
    output_crs,
    always_xy=True
)

x, y = transformer.transform(lon_geo, lat_geo)

valid = (
    np.isfinite(x) &
    np.isfinite(y) &
    (lon_geo >= -180) & (lon_geo <= 180) &
    (lat_geo >= -90) & (lat_geo <= 90)
)

geo_bounds = (
    float(np.min(x[valid])),
    float(np.min(y[valid])),
    float(np.max(x[valid])),
    float(np.max(y[valid]))
)

print("Geolocation bounds (UTM 17N):", geo_bounds)
print("Geolocation dimensions:", lon_geo.shape)

geo_profile = {
    "driver": "GTiff",
    "height": lon_geo.shape[0],
    "width": lon_geo.shape[1],
    "count": 2,
    "dtype": "float32",
    "compress": "LZW",
    "tiled": True,
    "BIGTIFF": "IF_SAFER"
}

with rasterio.open(geolocation_file, "w", **geo_profile) as dst:
    dst.write(lon_geo, 1)
    dst.write(lat_geo, 2)

print(f"Created geolocation grid: {lon_geo.shape}")

# Release large geolocation arrays
del lon_full, lat_full
del lon, lat, lon_geo, lat_geo

lon_ds = None
lat_ds = None


# --------------------------------------------------
# Create three-band in-memory displacement raster
# --------------------------------------------------
band_names = ["X", "Y", "SNR"]

factors = {
    "X": range_factor,
    "Y": azimuth_factor,
    "SNR": 1.0
}

mem = gdal.GetDriverByName("MEM").Create(
    "",
    width,
    height,
    len(band_names),
    gdal.GDT_Float32
)

for i, name in enumerate(band_names, start=1):

    if name not in offsets:
        continue

    data = offsets[name]
    factor = factors[name]

    if data.shape != (height, width):
        raise ValueError(
            f"{name} dimensions {data.shape} do not match "
            f"reference dimensions {(height, width)}"
        )

    valid = np.isfinite(data)

    src_nodata = offset_nodata.get(name)

    if src_nodata is not None:
        valid &= data != src_nodata

    output = np.full(
        (height, width),
        nodata,
        dtype=np.float32
    )

    output[valid] = data[valid] * factor

    band = mem.GetRasterBand(i)
    band.WriteArray(output)
    band.SetNoDataValue(nodata)

    print(f"Prepared band {i}: {name}")


# --------------------------------------------------
# Attach original subsampled geolocation convention
# --------------------------------------------------
mem.SetMetadata({
    "X_DATASET": geolocation_file,
    "X_BAND": "1",
    "Y_DATASET": geolocation_file,
    "Y_BAND": "2",
    "SRS": "EPSG:4326",
    "PIXEL_OFFSET": "0",
    "LINE_OFFSET": "0",
    "PIXEL_STEP": str(step),
    "LINE_STEP": str(step),
    "GEOREFERENCING_CONVENTION": "PIXEL_CENTER"
}, "GEOLOCATION")


# --------------------------------------------------
# Geocode all bands in one GDAL operation
# --------------------------------------------------
combined_file = f"{output_dir}/displacements_snr_utm17_temp.tif"

print("Geocoding all bands...")

warp_options = gdal.WarpOptions(
    format="GTiff",
    geoloc=True,
    dstSRS=output_crs,
    outputBounds=geo_bounds,
    xRes=resolution,
    yRes=resolution,
    targetAlignedPixels=True,
    resampleAlg="nearest",
    srcNodata=nodata,
    dstNodata=nodata,
    multithread=True,
    warpMemoryLimit=1024,
    errorThreshold=0.0,
    warpOptions=["NUM_THREADS=ALL_CPUS"],
    creationOptions=[
        "COMPRESS=LZW",
        "TILED=YES",
        "BIGTIFF=IF_SAFER"
    ]
)

gdal.Warp(
    combined_file,
    mem,
    options=warp_options
)

mem = None

print(f"Created: {combined_file}")


# --------------------------------------------------
# Split geocoded bands into individual output files
# --------------------------------------------------
output_names = {
    1: "range_displacement_utm17.tif",
    2: "azimuth_displacement_utm17.tif",
    3: "snr_utm17.tif"
}

for band, filename in output_names.items():

    # Skip SNR for MicMac
    if band == 3 and "SNR" not in offsets:
        continue

    output_file = f"{output_dir}/{filename}"

    gdal.Translate(
        output_file,
        combined_file,
        bandList=[band],
        creationOptions=[
            "COMPRESS=LZW",
            "TILED=YES",
            "BIGTIFF=IF_SAFER"
        ]
    )

    print(f"Saved: {output_file}")


# --------------------------------------------------
# Clean up temporary multiband output
# --------------------------------------------------
os.remove(combined_file)

print("Finished geocoding.")

RasterioIOError: /Users/chanagan/Downloads/Panama2026/NISAR_ASF/micmac/..._X.tif: No such file or directory